In [12]:
import torch

print("CUDA tersedia:", torch.cuda.is_available())
print("Nama GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Tidak ada GPU")

CUDA tersedia: True
Nama GPU: NVIDIA GeForce RTX 3060 Laptop GPU


In [13]:
import os, shutil, random, cv2, numpy as np, requests, time
from tqdm import tqdm

DETECTOR_DATASET = '../data/detector_dataset'
IMAGES_TRAIN = os.path.join(DETECTOR_DATASET, 'images', 'train')
IMAGES_VAL   = os.path.join(DETECTOR_DATASET, 'images', 'val')
LABELS_TRAIN = os.path.join(DETECTOR_DATASET, 'labels', 'train')
LABELS_VAL   = os.path.join(DETECTOR_DATASET, 'labels', 'val')

for d in [IMAGES_TRAIN, IMAGES_VAL, LABELS_TRAIN, LABELS_VAL]:
    os.makedirs(d, exist_ok=True)
print("✅ Folder detector_dataset siap.")

✅ Folder detector_dataset siap.


In [14]:
# Sel 2: Cek dataset_clean dan kumpulkan gambar positif
CLEAN_ASLI = '../data/dataset_clean/asli'
CLEAN_PALSU = '../data/dataset_clean/palsu'
if not os.path.exists(CLEAN_ASLI) or not os.path.exists(CLEAN_PALSU):
    raise RuntimeError("Jalankan prepare_clean_dataset() dari train_and_verify.ipynb dulu.")
positive_images = []
for folder in [CLEAN_ASLI, CLEAN_PALSU]:
    for f in os.listdir(folder):
        if f.lower().endswith(('.jpg','.jpeg','.png')):
            positive_images.append(os.path.join(folder, f))
print(f"Total gambar positif: {len(positive_images)}")
random.seed(42)
random.shuffle(positive_images)
split_idx = int(0.8 * len(positive_images))
train_pos = positive_images[:split_idx]
val_pos = positive_images[split_idx:]
print(f"Train positif: {len(train_pos)}, Val positif: {len(val_pos)}")

def create_full_frame_label(image_path, label_dir):
    base = os.path.splitext(os.path.basename(image_path))[0]
    with open(os.path.join(label_dir, base + '.txt'), 'w') as f:
        f.write('0 0.5 0.5 1.0 1.0\n')

def copy_and_label(src_list, dst_img_dir, dst_lbl_dir):
    for src in tqdm(src_list, desc=f'Copying to {os.path.basename(dst_img_dir)}'):
        dst_img = os.path.join(dst_img_dir, os.path.basename(src))
        shutil.copy2(src, dst_img)
        create_full_frame_label(src, dst_lbl_dir)

copy_and_label(train_pos, IMAGES_TRAIN, LABELS_TRAIN)
copy_and_label(val_pos, IMAGES_VAL, LABELS_VAL)
print("✅ Data positif disalin dan label dibuat.")

Total gambar positif: 1911
Train positif: 1528, Val positif: 383


Copying to train:   0%|          | 0/1528 [00:00<?, ?it/s]

Copying to val: 100%|██████████| 383/383 [00:00<00:00, 413.49it/s]

✅ Data positif disalin dan label dibuat.


In [15]:
# Sel 3 (Langkah 0.3 revisi): Unduh negatif dari Lorem Picsum
NUM_NEGATIVE = 200
TRAIN_COUNT = int(NUM_NEGATIVE * 0.8)
VAL_COUNT = NUM_NEGATIVE - TRAIN_COUNT

def download_random_image(url, save_path):
    try:
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        with open(save_path, 'wb') as f:
            f.write(resp.content)
        img = cv2.imread(save_path)
        if img is None:
            os.remove(save_path)
            return False
        return True
    except Exception as e:
        print(f"Gagal unduh {url}: {e}")
        return False

BASE_URL = "https://picsum.photos/640/480"
collected = 0
for i in range(NUM_NEGATIVE):
    fname = f"lorem_{i:04d}.jpg"
    dest = IMAGES_TRAIN if i < TRAIN_COUNT else IMAGES_VAL
    filepath = os.path.join(dest, fname)
    if download_random_image(BASE_URL, filepath):
        collected += 1
    else:
        # retry
        time.sleep(0.2)
        if download_random_image(BASE_URL + f"?random={i}", filepath):
            collected += 1
print(f"✅ {collected} gambar negatif tersimpan.")
# Sel pembagi lama sudah dihapus – tidak ada lagi.

✅ 200 gambar negatif tersimpan.


In [16]:
# Sel 4: Verifikasi (Langkah 0.4)
def verify_split(img_dir, lbl_dir):
    img_files = [f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    print(f"{img_dir}: {len(img_files)} gambar")
    for f in img_files:
        path = os.path.join(img_dir, f)
        if cv2.imread(path) is None:
            print(f"  ❌ Corrupt: {f}")
        base, ext = os.path.splitext(f)
        if f.startswith('lorem'):
            if os.path.exists(os.path.join(lbl_dir, base+'.txt')):
                print(f"  ⚠️ Negatif punya label: {f}")
        else:
            if not os.path.exists(os.path.join(lbl_dir, base+'.txt')):
                print(f"  ❌ Positif tanpa label: {f}")
    print("✅ Verifikasi selesai.")

verify_split(IMAGES_TRAIN, LABELS_TRAIN)
verify_split(IMAGES_VAL, LABELS_VAL)

train_set = set(os.listdir(IMAGES_TRAIN))
val_set = set(os.listdir(IMAGES_VAL))
overlap = train_set.intersection(val_set)
if overlap:
    print(f"❌ Overlap: {overlap}")
else:
    print("✅ Tidak ada overlap train/val.")

../data/detector_dataset\images\train: 1688 gambar
✅ Verifikasi selesai.
../data/detector_dataset\images\val: 423 gambar
✅ Verifikasi selesai.
✅ Tidak ada overlap train/val.


In [17]:
# Sel 5: Dokumentasi (Langkah 0.5)
readme = f"""# Detector Dataset for Medicine Package
...
Jumlah Data:
- Train positif: {len(train_pos)}
- Val positif: {len(val_pos)}
- Train negatif: {TRAIN_COUNT}
- Val negatif: {VAL_COUNT}
...
"""
with open(os.path.join(DETECTOR_DATASET, 'README.md'), 'w') as f:
    f.write(readme)
print("✅ README.md dibuat.")

✅ README.md dibuat.
